# SLM Labs — Thai Sign Language → Natural Thai → Speech (Demo 1)

Pipeline ที่ build ใน notebook นี้

```
TSL video ─► MediaPipe Holistic ─► Hand / Body / Face features (T×D)
          ─► Modality MLP encoders ─► Fusion ─► Transformer Encoder
          ─► [Classification head (CE)]  +  [CTC head (sign sequence)]
          ─► sign glosses + face cue ─► LLM (OpenAI GPT‑5.6) ─► natural Thai
          ─► Thai TTS (MMS‑TTS open-source / OpenAI TTS) ─► .wav
```

| Component | Choice (Demo 1) |
|---|---|
| Dataset | `Namonpas/thai-sign-language-tsl51` (โหลดด้วย `huggingface_hub.snapshot_download` ไม่ใช้ `load_dataset`) |
| Landmarks | MediaPipe Holistic — 21+21 hand, 6 pose, 6 face จุด (162 coords/frame, ~10 fps) |
| Feature Encoder | per-modality MLP (Hand / Body / Face) → projection → fusion |
| Temporal model | Pre-LN Transformer Encoder (GELU, learned positional embedding) |
| Sequence head | CTC (blank + 52 classes) + auxiliary classification head |
| LLM | OpenAI Responses API, `gpt-5.6` (ว่าง API key ได้ → fallback rule-based) |
| TTS | `facebook/mms-tts-tha` (open-source) หรือ OpenAI `gpt-4o-mini-tts` |

Metrics: Accuracy / Macro-F1 (isolated) · WER / CER (sequence) · BLEU / chrF (translation) · Face ablation · MOS template (TTS) · Semantic accuracy template (E2E) · Latency / FPS.

> รันตามลำดับจากบนลงล่าง — cell สุดท้ายคือ **inference ด้วยวิดีโอ .mp4 ของคุณเอง**

## 0. Setup

In [1]:
# env (slm_lab)
!pip install -q "torch>=2.3" "mediapipe>=0.10.14" opencv-python-headless "huggingface_hub>=0.24" \
                pandas numpy scikit-learn jiwer sacrebleu matplotlib tqdm openai transformers scipy

In [ ]:
import os, re, io, json, math, time, random, zipfile, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from dotenv import load_dotenv
import os

load_dotenv()


warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

@dataclass
class Config:
    # ---------- data ----------
    HF_REPO: str = "Namonpas/thai-sign-language-tsl51"
    DATA_DIR: Path = Path("./data/tsl51")
    USE_EXPERT_PRIMARY: bool = True      # expert_primary_02/03 (เฉพาะคลิป original, is_augmented == False)
    USE_EXPERT_AUGMENTED: bool = False   # True = ใช้ augmented 32k คลิปด้วย (ช้า, กิน RAM)
    N_SAMPLE_VIDEOS: int = 3             # โหลดวิดีโอตัวอย่างจาก dataset เพื่อ sanity-check MediaPipe extractor
    # ---------- features ----------
    TARGET_FPS: float = 10.0             # dataset landmarks ถูกดึงที่ ~10 fps (ดูจาก t_ms) → วิดีโอใหม่ต้อง resample ให้ตรง
    MAX_FRAMES_ISO: int = 96             # isolated sign
    MAX_FRAMES_SEQ: int = 256            # continuous sentence
    # ---------- model ----------
    D_MODEL: int = 256
    N_HEADS: int = 4
    N_LAYERS: int = 4
    D_FF: int = 512
    DROPOUT: float = 0.2
    # ---------- training ----------
    EPOCHS_A: int = 40                   # Stage A: isolated sign recognition (CE)
    EPOCHS_B: int = 40                   # Stage B: continuous (CTC + CE)
    BATCH_SIZE: int = 32
    LR: float = 3e-4
    WEIGHT_DECAY: float = 0.05
    LABEL_SMOOTH: float = 0.1
    CE_WEIGHT_STAGE_B: float = 0.3       # λ ของ CE loss ระหว่าง stage B
    SEED: int = 42
    # ---------- LLM ----------
    LLM_PROVIDER: str = "openai"         # "openai" | "rule" (ไม่เรียก API)
    LLM_MODEL: str = "gpt-5.6"
    OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
    # ---------- TTS ----------
    TTS_BACKEND: str = "mms"             # "mms" (facebook/mms-tts-tha, open-source) | "openai"
    OPENAI_TTS_MODEL: str = "gpt-4o-mini-tts"
    OPENAI_TTS_VOICE: str = "alloy"
    # ---------- output ----------
    OUT_DIR: Path = Path("./outputs")

cfg = Config()
cfg.OUT_DIR.mkdir(parents=True, exist_ok=True)
if cfg.OPENAI_API_KEY: os.environ["OPENAI_API_KEY"] = cfg.OPENAI_API_KEY

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(cfg.SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)
print(json.dumps({k: str(v) for k, v in asdict(cfg).items() if "KEY" not in k}, ensure_ascii=False, indent=1))

## 1. Input — ดึง dataset จาก Hugging Face (ไม่ใช้ `load_dataset`)

โหลดเฉพาะ `metadata/`, `landmarks/user_sign`, `landmarks/user_sentence` และ expert zip (ถ้าเปิด) — ไม่โหลดวิดีโอทั้งหมด (2.5 GB) ยกเว้นตัวอย่างไม่กี่ไฟล์เพื่อทดสอบ extractor

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

allow = ["README.md", "metadata/*", "landmarks/user_sign/*", "landmarks/user_sentence/*"]
if cfg.USE_EXPERT_PRIMARY or cfg.USE_EXPERT_AUGMENTED:
    allow += ["landmarks/expert_primary_02.zip", "landmarks/expert_primary_03.zip"]

local = snapshot_download(repo_id=cfg.HF_REPO, repo_type="dataset",
                          local_dir=str(cfg.DATA_DIR), allow_patterns=allow)
DATA = Path(local)
print("dataset at:", DATA)
for p in sorted(DATA.rglob("*")):
    if p.is_dir(): 
        n = len(list(p.glob("*"))); print(f"  {p.relative_to(DATA)}/  ({n} items)")

In [ ]:
# metadata
meta_sign = pd.read_csv(DATA / "metadata/user_sign_metadata.csv")
meta_sent = pd.read_csv(DATA / "metadata/sentence_metadata.csv")
meta_exp  = pd.read_csv(DATA / "metadata/expert_metadata.csv") if (DATA/"metadata/expert_metadata.csv").exists() else None
print("user_sign:", meta_sign.shape, "| sentence:", meta_sent.shape, "| expert:", None if meta_exp is None else meta_exp.shape)
display(meta_sign.head(3)); display(meta_sent.head(3))
if meta_exp is not None: display(meta_exp.head(3))

In [ ]:
# วิดีโอตัวอย่าง (ประโยค) สำหรับ sanity-check ว่า MediaPipe extractor ของเราให้ค่าเหมือน landmark ใน dataset
sample_videos = []
vcol = "video_path" if "video_path" in meta_sent.columns else None
cands = meta_sent[vcol].tolist() if vcol else []
for rel in cands[: cfg.N_SAMPLE_VIDEOS]:
    rel = rel.replace("\\", "/")
    if not rel.startswith("videos/"): rel = "videos/user_sentence/" + Path(rel).name
    try:
        p = hf_hub_download(cfg.HF_REPO, rel, repo_type="dataset", local_dir=str(cfg.DATA_DIR))
        sample_videos.append(Path(p))
    except Exception as e:
        print("skip", rel, "->", type(e).__name__)
print("sample videos:", [p.name for p in sample_videos])

## 2. ดูข้อมูลอธิบายประกอบ dataset (EDA)

In [ ]:
# 52 classes (51 signs + null_act) จาก README — ใช้เป็น vocabulary หลัก
SIGN_CLASSES = [
 "แม่_var_1","คุณ_var_1","แฟน_var_1","พี่_var_1","น้อง_var_1","ฉัน_var_2","ฉัน_var_1","ยาย_var_1","แมว_var_1",
 "ภาษามือ_var_1","ขนมปัง_var_1","ข้าว_var_1","ไข่_var_2",
 "ถาม_ทำท่ามือถามไปยังผู้นั้น","เรียก_var_1","ชอบ_var_1","เที่ยว_var_1","ไป_var_2",
 "เหงา_var_1","แต่งงาน_var_1","ร้อน_สองมือ","โสด_var_1","โกรธ_var_1","กลัว_var_1","เหนื่อย_var_1","สบายดี_var_1",
 "ทำงาน_var_2","เกิด_var_2","ง่วง_var_1","อยู่บ้าน_var_1",
 "กิน_var_1","เรียน_var_1","ชื่อ_var_1","ดี_var_1","ด้วยกัน_var_1","หูหนวก_var_2",
 "สวัสดี_อายุเท่ากันหรือน้อยกว่า","ขอโทษ_อายุเท่ากันหรือน้อยกว่า","ขอบคุณ_เปิดมือสองข้าง",
 "วันนี้_var_1","เช้า_var_1","วันหยุด_var_1","พรุ่งนี้_var_1","อย่า_var_1","ทำไม_var_1","อะไร_var_1","ที่ไหน_var_1",
 "กรุงเทพ_var_1","โรงเรียน_var_1","บ้าน_var_1","ตลาด_var_4","null_act",
]
assert len(SIGN_CLASSES) == 52
NULL_CLASS = "null_act"

def class_to_word(c):  # "ฉัน_var_1" -> "ฉัน", "ขอบคุณ_เปิดมือสองข้าง" -> "ขอบคุณ"
    if c == NULL_CLASS: return "<null>"
    return re.split(r"_(var_\d+|สอง|อายุ|เปิด|ทำท่า)", c)[0]

CLASS2WORD = {c: class_to_word(c) for c in SIGN_CLASSES}
WORD2CLASSES = defaultdict(list)
for c, w in CLASS2WORD.items(): WORD2CLASSES[w].append(c)

# ตรวจว่า sign_id ใน metadata ตรงกับ vocab
sid_col = "sign_id" if "sign_id" in meta_sign.columns else meta_sign.columns[1]
unknown = sorted(set(meta_sign[sid_col]) - set(SIGN_CLASSES))
print("classes in user_sign metadata:", meta_sign[sid_col].nunique(), "| not in vocab:", unknown)

In [ ]:
# การกระจายของ class (isolated) และ pattern ของประโยค
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
meta_sign[sid_col].value_counts().plot.bar(ax=ax[0]); ax[0].set_title("user_sign: videos per class"); ax[0].tick_params(labelsize=6)

SUFFIX_RE = re.compile(r"_(no_space|with_space)_\d+$")
def parse_sentence_glosses(stem: str):
    "'กรุงเทพ_var_1__ฉัน_var_1__เที่ยว_var_1__ชอบ_var_1_no_space_0' -> ['กรุงเทพ_var_1', 'ฉัน_var_1', ...]"
    stem = SUFFIX_RE.sub("", Path(stem).stem)
    toks = [t for t in stem.split("__") if t]
    out = []
    for t in toks:
        if t in SIGN_CLASSES: out.append(t); continue
        t2 = re.sub(r"_\d+$", "", t)                       # เผื่อไม่มี suffix no_space แต่มีเลขท้าย เช่น กิน_var_1_3
        if t2 in SIGN_CLASSES: out.append(t2); continue
        w = class_to_word(t)  # variant ไม่ตรง → map ด้วยคำ (เช่น ไป_var_1 -> ไป_var_2)
        out.append(WORD2CLASSES[w][0] if w in WORD2CLASSES else t)
    return out

lp_col = "landmark_path" if "landmark_path" in meta_sent.columns else meta_sent.columns[-2]
meta_sent["glosses"] = meta_sent[lp_col].map(parse_sentence_glosses)
meta_sent["pattern"] = meta_sent["glosses"].map(lambda g: " ".join(CLASS2WORD.get(x, x) for x in g))
bad = meta_sent[meta_sent["glosses"].map(lambda g: any(x not in SIGN_CLASSES for x in g))]
print("sentences:", len(meta_sent), "| unique patterns:", meta_sent["pattern"].nunique(), "| unparsable:", len(bad))
meta_sent["pattern"].value_counts().head(20).plot.barh(ax=ax[1]); ax[1].set_title("top sentence patterns"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
print("words per sentence:", meta_sent["glosses"].map(len).describe()[["min","mean","max"]].round(2).to_dict())

In [ ]:
# โครงสร้าง landmark CSV + fps ที่ใช้จริง
LH_COLS  = [f"lh_{a}{i}" for i in range(21) for a in "xyz"]
RH_COLS  = [f"rh_{a}{i}" for i in range(21) for a in "xyz"]
POSE_PTS = ["l_shoulder","r_shoulder","l_elbow","r_elbow","l_wrist","r_wrist"]
POSE_COLS= [f"{p}_{a}" for p in POSE_PTS for a in "xyz"]
FACE_PTS = ["lbrow_outer","lbrow_inner","rbrow_inner","rbrow_outer","mouth_right","mouth_left"]
FACE_COLS= [f"{p}_{a}" for p in FACE_PTS for a in "xyz"]
ALL_COLS = ["frame","t_ms"] + LH_COLS + RH_COLS + \
           [f"{p}_{a}" for p in POSE_PTS for a in ["x","y","z","vis","pres"]] + FACE_COLS

def resolve_landmark(rel):
    rel = str(rel).replace("\\", "/")
    p = DATA / rel
    if p.exists(): return p
    for sub in ["landmarks/user_sign", "landmarks/user_sentence"]:
        q = DATA / sub / Path(rel).name
        if q.exists(): return q
    return None

ex = pd.read_csv(resolve_landmark(meta_sent[lp_col].iloc[0]))
print("columns:", len(ex.columns), "| frames:", len(ex))
print("missing expected cols:", [c for c in ALL_COLS if c not in ex.columns][:10])
dt = np.diff(ex["t_ms"].values)
print(f"median Δt = {np.median(dt):.1f} ms  →  ≈ {1000/np.median(dt):.1f} fps  (cfg.TARGET_FPS = {cfg.TARGET_FPS})")
print("NaN ratio  lh/rh/pose/face:", 
      round(ex[LH_COLS].isna().mean().mean(),3), round(ex[RH_COLS].isna().mean().mean(),3),
      round(ex[POSE_COLS].isna().mean().mean(),3), round(ex[FACE_COLS].isna().mean().mean(),3))
display(ex.iloc[:3, :8])

# ความยาว sequence
lens_sign = [len(pd.read_csv(resolve_landmark(p), usecols=["frame"])) for p in meta_sign["landmark_path"].iloc[:200]]
lens_sent = [len(pd.read_csv(resolve_landmark(p), usecols=["frame"])) for p in meta_sent[lp_col]]
plt.figure(figsize=(10,3)); plt.hist(lens_sign, 30, alpha=.6, label="isolated"); plt.hist(lens_sent, 30, alpha=.6, label="sentence")
plt.legend(); plt.title("frames per clip"); plt.show()
print("isolated frames:", np.percentile(lens_sign,[5,50,95]).round(0), "| sentence frames:", np.percentile(lens_sent,[5,50,95]).round(0))

## 3. Data Processing + Feature Engineering

- **normalize** ทุกจุดโดยอ้างอิง body: ลบด้วยจุดกึ่งกลางไหล่ แล้วหารด้วยความกว้างไหล่ (scale/translation invariant)
- **Hand** `H_t` = 2 มือ × 21 จุด × (x,y,z) + Δ(x,y,z) + presence flag → 254 dims
- **Body** `B_t` = pose 6 จุด xyz + relative vectors (elbow/wrist − shoulder, wrist−wrist) + velocity → 51 dims
- **Face** `F_t` = 6 จุด xyz + Δ + geometry (brow-mouth distance, mouth width, brow gap) → 39 dims
- NaN (ไม่เจอมือ) → 0 + presence = 0 ; pose/face NaN → interpolate ตามเวลา

In [ ]:
def _ffill_bfill(a):  # a: (T, N, 3) interpolate NaN along time
    T = a.shape[0]; out = a.copy().reshape(T, -1)
    df = pd.DataFrame(out).interpolate(limit_direction="both", axis=0)
    return df.fillna(0.0).values.reshape(a.shape)

def load_landmarks(src) -> dict:
    "src: path | DataFrame → dict(lh, rh, pose, face, t_ms) with shapes (T,21,3),(T,21,3),(T,6,3),(T,6,3)"
    df = src if isinstance(src, pd.DataFrame) else pd.read_csv(src)
    for c in ALL_COLS:
        if c not in df.columns: df[c] = np.nan
    T = len(df)
    return dict(
        lh   = df[LH_COLS].values.reshape(T, 21, 3).astype(np.float32),
        rh   = df[RH_COLS].values.reshape(T, 21, 3).astype(np.float32),
        pose = df[POSE_COLS].values.reshape(T, 6, 3).astype(np.float32),
        face = df[FACE_COLS].values.reshape(T, 6, 3).astype(np.float32),
        t_ms = df["t_ms"].values.astype(np.float32),
    )

HAND_DIM, BODY_DIM, FACE_DIM = 21*3*2*2 + 2, 6*3 + 5*3 + 6*3, 6*3 + 6*3 + 3

def make_features(lm: dict):
    "→ dict(hand (T,254), body (T,51), face (T,39))"
    pose = _ffill_bfill(lm["pose"]); face = _ffill_bfill(lm["face"])
    ls, rs = pose[:, 0], pose[:, 1]
    center = (ls + rs) / 2                                             # (T,3)
    width  = np.linalg.norm((ls - rs)[:, :2], axis=-1)                 # (T,)
    width  = np.where(np.isfinite(width) & (width > 1e-3), width, np.nanmedian(width) if np.isfinite(np.nanmedian(width)) else 1.0)
    norm = lambda p: (p - center[:, None, :]) / width[:, None, None]

    # ---- hands
    xyz, vel_, pres = [], [], []
    for h in (lm["lh"], lm["rh"]):
        present = (~np.isnan(h[:, 0, 0])).astype(np.float32)          # (T,)
        hn = norm(np.nan_to_num(h, nan=0.0)) * present[:, None, None]
        vel = np.diff(hn, axis=0, prepend=hn[:1]) * present[:, None, None]
        xyz.append(hn.reshape(len(hn), -1)); vel_.append(vel.reshape(len(hn), -1)); pres.append(present[:, None])
    hand = np.concatenate(xyz + vel_ + pres, axis=1)                  # layout: [lh xyz 63][rh xyz 63][lh Δ 63][rh Δ 63][lh pres][rh pres]

    # ---- body
    pn = norm(pose)                                                    # (T,6,3)
    rel = np.stack([pn[:,2]-pn[:,0], pn[:,4]-pn[:,0], pn[:,3]-pn[:,1], pn[:,5]-pn[:,1], pn[:,4]-pn[:,5]], 1)
    pvel = np.diff(pn, axis=0, prepend=pn[:1])
    body = np.concatenate([pn.reshape(len(pn),-1), rel.reshape(len(pn),-1), pvel.reshape(len(pn),-1)], 1)

    # ---- face
    fn = norm(face)                                                    # (T,6,3)
    fvel = np.diff(fn, axis=0, prepend=fn[:1])
    brow_y  = fn[:, :4, 1].mean(1); mouth_y = fn[:, 4:, 1].mean(1)
    geo = np.stack([mouth_y - brow_y,                                  # brow raise (ยิ่งมาก = คิ้วยกสูง)
                    np.linalg.norm(fn[:,4,:2]-fn[:,5,:2], axis=-1),    # mouth width
                    np.linalg.norm(fn[:,1,:2]-fn[:,2,:2], axis=-1)], 1)# inner-brow gap
    facef = np.concatenate([fn.reshape(len(fn),-1), fvel.reshape(len(fn),-1), geo], 1)

    return dict(hand=np.nan_to_num(hand).astype(np.float32),
                body=np.nan_to_num(body).astype(np.float32),
                face=np.nan_to_num(facef).astype(np.float32))

feat = make_features(load_landmarks(resolve_landmark(meta_sent[lp_col].iloc[0])))
print({k: v.shape for k, v in feat.items()}, "| expected dims:", HAND_DIM, BODY_DIM, FACE_DIM)
assert feat["hand"].shape[1]==HAND_DIM and feat["body"].shape[1]==BODY_DIM and feat["face"].shape[1]==FACE_DIM

plt.figure(figsize=(12,3)); plt.plot(feat["hand"][:, -2], label="LH present"); plt.plot(feat["hand"][:, -1], label="RH present")
plt.plot(feat["face"][:, -3]*5, label="brow-raise ×5"); plt.legend(); plt.title("example sentence clip"); plt.show()

## 4. สร้างชุดข้อมูล (isolated + continuous) และ split

In [ ]:
CLS2ID = {c: i for i, c in enumerate(SIGN_CLASSES)}
BLANK = 0                                   # CTC blank ; class i → token i+1
N_CLASSES = len(SIGN_CLASSES)

def read_expert_from_zip(max_per_class=None):
    "อ่าน landmark CSV ของ expert (original เท่านั้น ยกเว้นตั้ง USE_EXPERT_AUGMENTED) ตรงจาก zip โดยไม่แตกไฟล์"
    if meta_exp is None: return []
    m = meta_exp.copy()
    if "source_group" in m.columns: m = m[m["source_group"].astype(str).str.contains("primary", case=False)]
    if "is_augmented" in m.columns and not cfg.USE_EXPERT_AUGMENTED:
        m = m[m["is_augmented"].astype(str).str.upper() != "TRUE"]
    zips = [zipfile.ZipFile(p) for p in DATA.glob("landmarks/expert_primary_*.zip")]
    index = {}
    for z in zips:
        for n in z.namelist():
            if n.endswith(".csv"): index[Path(n).name] = (z, n)
    out, per = [], Counter()
    for _, r in tqdm(m.iterrows(), total=len(m), desc="expert csv"):
        sid = r["sign_id"]
        if sid not in CLS2ID: continue
        if max_per_class and per[sid] >= max_per_class: continue
        key = Path(str(r["landmark_path"]).replace("\\","/")).name
        if key not in index: continue
        z, n = index[key]
        df = pd.read_csv(io.BytesIO(z.read(n)))
        out.append(dict(feat=make_features(load_landmarks(df)), label=CLS2ID[sid], src="expert", id=str(r.get("video_id", key))))
        per[sid] += 1
    print(f"expert clips loaded: {len(out)} over {len(per)} classes")
    return out

def build_isolated():
    items = []
    for _, r in tqdm(meta_sign.iterrows(), total=len(meta_sign), desc="user_sign csv"):
        p = resolve_landmark(r["landmark_path"])
        if p is None or r[sid_col] not in CLS2ID: continue
        items.append(dict(feat=make_features(load_landmarks(p)), label=CLS2ID[r[sid_col]], src="user", id=str(r["video_id"])))
    return items

def build_sentences():
    items = []
    for _, r in tqdm(meta_sent.iterrows(), total=len(meta_sent), desc="user_sentence csv"):
        p = resolve_landmark(r[lp_col])
        if p is None or any(g not in CLS2ID for g in r["glosses"]): continue
        items.append(dict(feat=make_features(load_landmarks(p)), labels=[CLS2ID[g]+1 for g in r["glosses"]],
                          glosses=r["glosses"], pattern=r["pattern"], id=str(r["video_id"]), path=str(p)))
    return items

iso_user = build_isolated()
iso_expert = read_expert_from_zip() if cfg.USE_EXPERT_PRIMARY else []
sent_all = build_sentences()
print(f"isolated user={len(iso_user)}  expert={len(iso_expert)} | sentences={len(sent_all)}")

In [ ]:
from sklearn.model_selection import train_test_split

# isolated: user_sign → 70/15/15 stratified ; expert → train เท่านั้น (คนละ signer ช่วย generalization)
y_user = [it["label"] for it in iso_user]
idx_tr, idx_tmp = train_test_split(range(len(iso_user)), test_size=0.30, stratify=y_user, random_state=cfg.SEED)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, stratify=[y_user[i] for i in idx_tmp], random_state=cfg.SEED)
iso_train = [iso_user[i] for i in idx_tr] + iso_expert
iso_val   = [iso_user[i] for i in idx_va]
iso_test  = [iso_user[i] for i in idx_te]

# sentences: แบ่งตาม pattern → ทุก pattern ที่มี ≥3 คลิป จะมี 1 คลิปใน val หรือ test (สลับกัน)
rng = random.Random(cfg.SEED); by_pat = defaultdict(list)
for it in sent_all: by_pat[it["pattern"]].append(it)
sent_train, sent_val, sent_test = [], [], []
for k, (pat, its) in enumerate(sorted(by_pat.items())):
    rng.shuffle(its)
    if len(its) >= 3:   sent_train += its[2:]; sent_val.append(its[0]); sent_test.append(its[1])
    elif len(its) == 2: sent_train.append(its[0]); (sent_val if k % 2 else sent_test).append(its[1])
    else:               sent_train += its
print(f"isolated  train/val/test = {len(iso_train)}/{len(iso_val)}/{len(iso_test)}")
print(f"sentence  train/val/test = {len(sent_train)}/{len(sent_val)}/{len(sent_test)}")

In [ ]:
def temporal_resample(feat: dict, n_out: int):
    "uniform index resample ทุก modality ให้ยาว n_out"
    T = len(feat["hand"]); idx = np.linspace(0, T-1, n_out).round().astype(int)
    return {k: v[idx] for k, v in feat.items()}

AUG = dict(speed=(0.8, 1.25), rot_deg=12, scale=(0.85, 1.15), shift=0.05, hand_drop=0.15, mirror=0.3, noise=0.01)

def _xyz_blocks(f):
    "ตำแหน่ง (start, n_points) ของบล็อก x,y,z ใน feature vectors (ไม่รวม presence/geometry) เพื่อทำ spatial transform"
    return dict(hand=[(0,21),(63,21),(126,21),(189,21)], body=[(0,6),(18,5),(33,6)], face=[(0,6),(18,6)])

def spatial_transform(f: dict, deg, scale, dx, dy):
    "หมุน/ย่อขยาย/เลื่อน ทั้ง skeleton (จำลองมุมกล้องเอียง ระยะกล้อง การจัดกรอบ)"
    th = math.radians(deg); R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]], np.float32) * scale
    out = {}
    for k, v in f.items():
        v = v.copy()
        for st, n in _xyz_blocks(f)[k]:
            xy = v[:, st:st+3*n].reshape(len(v), n, 3)
            xy[..., :2] = xy[..., :2] @ R.T + np.array([dx, dy], np.float32)
            v[:, st:st+3*n] = xy.reshape(len(v), -1)
        out[k] = v
    return out

def mirror_swap(f: dict):
    "สะท้อนซ้าย-ขวา + สลับมือซ้าย/ขวา และ pose/face ซ้าย/ขวา (จำลองคนถนัดซ้าย / กล้องมิเรอร์)"
    h = f["hand"].copy()
    h[:, 0:63], h[:, 63:126] = f["hand"][:, 63:126], f["hand"][:, 0:63]          # xyz
    h[:, 126:189], h[:, 189:252] = f["hand"][:, 189:252], f["hand"][:, 126:189]  # Δ
    h[:, 252], h[:, 253] = f["hand"][:, 253], f["hand"][:, 252]                  # presence
    b = f["body"].copy()
    for st, n in _xyz_blocks(f)["body"]:
        blk = b[:, st:st+3*n].reshape(len(b), n, 3)
        blk = blk[:, [1,0,3,2,5,4]] if n == 6 else blk[:, [2,3,0,1,4]]           # pose l/r , rel vectors
        b[:, st:st+3*n] = blk.reshape(len(b), -1)
    fc = f["face"].copy()
    for st, n in _xyz_blocks(f)["face"]:
        fc[:, st:st+3*n] = fc[:, st:st+3*n].reshape(len(fc), n, 3)[:, [3,2,1,0,5,4]].reshape(len(fc), -1)
    out = dict(hand=h, body=b, face=fc)
    for k in out:                                                                  # flip x ของทุกบล็อก xyz/Δ
        for st, n in _xyz_blocks(f)[k]:
            out[k][:, st:st+3*n:3] *= -1
    out["body"][:, 30:33] *= -1                                                    # wrist-to-wrist vector เปลี่ยนทิศทั้งเวกเตอร์
    return out

def augment(feat: dict):
    "feature-level augmentation: temporal + spatial + hand-dropout + mirror (จำลองกล้อง/คน/สภาพจริง)"
    T = len(feat["hand"])
    f = temporal_resample(feat, max(8, int(T * random.uniform(*AUG["speed"]))))
    if random.random() < 0.5:                                                     # random crop
        T2 = len(f["hand"]); c = int(T2 * random.uniform(0.0, 0.1)); e = T2 - int(T2 * random.uniform(0.0, 0.1))
        f = {k: v[c:max(e, c+8)] for k, v in f.items()}
    f = spatial_transform(f, random.uniform(-AUG["rot_deg"], AUG["rot_deg"]), random.uniform(*AUG["scale"]),
                          random.uniform(-AUG["shift"], AUG["shift"]), random.uniform(-AUG["shift"], AUG["shift"]))
    if random.random() < AUG["mirror"]: f = mirror_swap(f)
    if random.random() < AUG["hand_drop"]:                                        # จำลอง MediaPipe จับมือไม่ได้บางเฟรม
        T2 = len(f["hand"]); s0 = random.randrange(T2); e0 = min(T2, s0 + random.randint(1, max(1, T2 // 6)))
        side = random.choice([(0, 126, 252), (126, 252, 253)])
        f["hand"][s0:e0, side[0]:side[1]] = 0; f["hand"][s0:e0, side[2]] = 0
    return {k: v + np.random.normal(0, AUG["noise"], v.shape).astype(np.float32) for k, v in f.items()}

class SignDataset(Dataset):
    def __init__(self, items, max_frames, train=False, seq=False):
        self.items, self.max_frames, self.train, self.seq = items, max_frames, train, seq
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]; f = it["feat"]
        if self.train: f = augment(f)
        if len(f["hand"]) > self.max_frames: f = temporal_resample(f, self.max_frames)
        tgt = it["labels"] if self.seq else it["label"]
        return f, tgt

def collate(batch):
    feats, tgts = zip(*batch)
    lens = torch.tensor([len(f["hand"]) for f in feats]); Tm = int(lens.max())
    def pad(k):
        D = feats[0][k].shape[1]; out = torch.zeros(len(feats), Tm, D)
        for i, f in enumerate(feats): out[i, :len(f[k])] = torch.from_numpy(f[k])
        return out
    mask = torch.arange(Tm)[None, :] >= lens[:, None]               # True = padding
    x = dict(hand=pad("hand"), body=pad("body"), face=pad("face"), mask=mask, lens=lens)
    if isinstance(tgts[0], list):
        tl = torch.tensor([len(t) for t in tgts]); x["targets"] = torch.tensor([t for s in tgts for t in s]); x["target_lens"] = tl
    else:
        x["targets"] = torch.tensor(tgts)
    return x

def make_loader(items, max_frames, train, seq, bs=None):
    return DataLoader(SignDataset(items, max_frames, train, seq), batch_size=bs or cfg.BATCH_SIZE,
                      shuffle=train, collate_fn=collate, num_workers=0, drop_last=False)

dl_iso_tr = make_loader(iso_train, cfg.MAX_FRAMES_ISO, True,  False)
dl_iso_va = make_loader(iso_val,   cfg.MAX_FRAMES_ISO, False, False)
dl_iso_te = make_loader(iso_test,  cfg.MAX_FRAMES_ISO, False, False)
dl_seq_tr = make_loader(sent_train,cfg.MAX_FRAMES_SEQ, True,  True, bs=16)
dl_seq_va = make_loader(sent_val,  cfg.MAX_FRAMES_SEQ, False, True, bs=16)
dl_seq_te = make_loader(sent_test, cfg.MAX_FRAMES_SEQ, False, True, bs=16)
b = next(iter(dl_seq_tr)); print({k: tuple(v.shape) for k, v in b.items()})

## 5. Multimodal Encoder (Main Model)

```
Hand ─► MLP ─┐
Body ─► MLP ─┼─► concat ─► Fusion (Linear+LN) ─► +PosEmb ─► Transformer Encoder (pre-LN, GELU)
Face ─► MLP ─┘                                                        │
                                                         ┌────────────┴────────────┐
                                                 masked mean-pool             per-frame
                                                 → cls head (CE)              → CTC head
                                         face MLP output → face_embedding (ส่งให้ LLM เป็น cue)
```
`use_face=False` = ablation (zero-out face branch)

In [ ]:
class ModalityMLP(nn.Module):
    def __init__(self, d_in, d, p):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, d), nn.LayerNorm(d))
    def forward(self, x): return self.net(x)

class SignEncoder(nn.Module):
    def __init__(self, n_classes, d=256, heads=4, layers=4, ff=512, p=0.2, max_len=1024):
        super().__init__()
        self.hand, self.body, self.face = ModalityMLP(HAND_DIM, d, p), ModalityMLP(BODY_DIM, d, p), ModalityMLP(FACE_DIM, d, p)
        self.fusion = nn.Sequential(nn.Linear(3*d, d), nn.LayerNorm(d), nn.Dropout(p))
        self.pos = nn.Embedding(max_len, d)
        layer = nn.TransformerEncoderLayer(d, heads, ff, p, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(d)
        self.cls_head = nn.Linear(d, n_classes)
        self.ctc_head = nn.Linear(d, n_classes + 1)          # +blank
        self.face_proj = nn.Linear(d, 64)                    # face embedding (compact)

    @staticmethod
    def masked_mean(x, mask):                                # mask True=pad
        keep = (~mask).unsqueeze(-1).float()
        return (x * keep).sum(1) / keep.sum(1).clamp(min=1)

    def forward(self, hand, body, face, mask, use_face=True):
        h, b, f = self.hand(hand), self.body(body), self.face(face)
        if not use_face: f = torch.zeros_like(f)
        x = self.fusion(torch.cat([h, b, f], -1))
        x = x + self.pos(torch.arange(x.size(1), device=x.device))[None]
        x = self.norm(self.encoder(x, src_key_padding_mask=mask))
        return dict(logits_cls=self.cls_head(self.masked_mean(x, mask)),
                    logits_ctc=self.ctc_head(x),                       # (B,T,V)
                    face_emb=self.face_proj(self.masked_mean(f, mask)),
                    frame_emb=x)

model = SignEncoder(N_CLASSES, cfg.D_MODEL, cfg.N_HEADS, cfg.N_LAYERS, cfg.D_FF, cfg.DROPOUT).to(DEVICE)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")
with torch.no_grad():
    o = model(b["hand"].to(DEVICE), b["body"].to(DEVICE), b["face"].to(DEVICE), b["mask"].to(DEVICE))
print({k: tuple(v.shape) for k, v in o.items()})

## 6. Metrics utilities (Accuracy / Macro-F1 / WER / CER / BLEU / chrF / CTC decode)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import jiwer, sacrebleu

def ctc_greedy_decode(logits_ctc, lens, drop_null=True):
    "logits (B,T,V) → list[list[class_id]]  (collapse repeats, remove blank, drop null_act)"
    pred = logits_ctc.argmax(-1).cpu().numpy(); out = []
    for p, L in zip(pred, lens):
        seq, prev = [], BLANK
        for t in p[:L]:
            if t != BLANK and t != prev: seq.append(int(t) - 1)
            prev = t
        if drop_null: seq = [c for c in seq if SIGN_CLASSES[c] != NULL_CLASS]
        out.append(seq)
    return out

def ids_to_words(ids): return [CLASS2WORD[SIGN_CLASSES[i]] for i in ids]

def wer_cer(refs_words, hyps_words):
    "refs/hyps: list of word lists → WER (word-level), CER (char-level on joined Thai)"
    r = [" ".join(x) for x in refs_words]; h = [" ".join(x) if x else "<empty>" for x in hyps_words]
    wer = jiwer.wer(r, h)
    cer = jiwer.cer(["".join(x) for x in refs_words], ["".join(x) if x else "?" for x in hyps_words])
    return wer, cer

def bleu_chrf(refs, hyps):
    "Thai ไม่มีช่องว่าง → ใช้ char-level BLEU + chrF"
    bleu = sacrebleu.corpus_bleu(hyps, [refs], tokenize="char").score
    chrf = sacrebleu.corpus_chrf(hyps, [refs]).score
    return bleu, chrf

@torch.no_grad()
def eval_isolated(model, dl, use_face=True):
    model.eval(); ys, ps = [], []
    for x in dl:
        o = model(x["hand"].to(DEVICE), x["body"].to(DEVICE), x["face"].to(DEVICE), x["mask"].to(DEVICE), use_face)
        ps += o["logits_cls"].argmax(-1).cpu().tolist(); ys += x["targets"].tolist()
    return dict(acc=accuracy_score(ys, ps), macro_f1=f1_score(ys, ps, average="macro"), y=ys, p=ps)

@torch.no_grad()
def eval_sequence(model, dl, use_face=True):
    model.eval(); refs, hyps = [], []
    for x in dl:
        o = model(x["hand"].to(DEVICE), x["body"].to(DEVICE), x["face"].to(DEVICE), x["mask"].to(DEVICE), use_face)
        hyps += [ids_to_words(s) for s in ctc_greedy_decode(o["logits_ctc"], x["lens"].tolist())]
        i = 0
        for L in x["target_lens"].tolist():
            refs.append(ids_to_words([t-1 for t in x["targets"][i:i+L].tolist()])); i += L
    wer, cer = wer_cer(refs, hyps)
    exact = np.mean([r == h for r, h in zip(refs, hyps)])
    return dict(wer=wer, cer=cer, sent_acc=exact, refs=refs, hyps=hyps)

## 7. Training — Model A (Sign Encoder)

**Stage A**: isolated sign recognition (Cross-Entropy + label smoothing) → **Stage B**: continuous sentences (CTC) + CE เสริม (จาก isolated) เพื่อไม่ให้ลืม

In [ ]:
def make_optim(model, epochs, steps_per_epoch):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY, betas=(0.9, 0.98))
    total = max(1, epochs * steps_per_epoch); warm = max(1, int(0.1 * total))          # warmup 10% + cosine decay
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    return opt, sch

use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler(enabled=use_amp)
ce_loss = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTH)

def to_dev(x): return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in x.items()}

def train_stage_a(model, epochs):
    opt, sch = make_optim(model, epochs, len(dl_iso_tr)); best, hist = -1, []
    for ep in range(1, epochs+1):
        model.train(); tot = 0
        for x in dl_iso_tr:
            x = to_dev(x)
            with torch.autocast(DEVICE, enabled=use_amp):
                o = model(x["hand"], x["body"], x["face"], x["mask"]); loss = ce_loss(o["logits_cls"], x["targets"])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
        va = eval_isolated(model, dl_iso_va); hist.append((ep, tot/len(dl_iso_tr), va["acc"], va["macro_f1"]))
        if va["macro_f1"] > best: best = va["macro_f1"]; torch.save(model.state_dict(), cfg.OUT_DIR/"stageA_best.pt")
        if ep % 5 == 0 or ep == 1: print(f"[A] ep{ep:3d} loss={tot/len(dl_iso_tr):.3f}  val acc={va['acc']:.3f}  macroF1={va['macro_f1']:.3f}")
    model.load_state_dict(torch.load(cfg.OUT_DIR/"stageA_best.pt", map_location=DEVICE)); return pd.DataFrame(hist, columns=["ep","loss","val_acc","val_f1"])

t0 = time.time(); histA = train_stage_a(model, cfg.EPOCHS_A); print(f"stage A done in {(time.time()-t0)/60:.1f} min")
histA.plot(x="ep", y=["val_acc","val_f1"], figsize=(8,3), title="Stage A"); plt.show()

In [ ]:
# ---- Stage A evaluation: Accuracy / Macro-F1 + confusion
resA = eval_isolated(model, dl_iso_te)
print(f"Stage A TEST  accuracy={resA['acc']:.4f}   macro-F1={resA['macro_f1']:.4f}")
cm = confusion_matrix(resA["y"], resA["p"], labels=range(N_CLASSES))
conf = [(SIGN_CLASSES[i], SIGN_CLASSES[j], int(cm[i,j])) for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j and cm[i,j] > 0]
print("top confusions:", sorted(conf, key=lambda t: -t[2])[:8])
plt.figure(figsize=(9,8)); plt.imshow(cm / cm.sum(1, keepdims=True).clip(min=1), cmap="Blues"); plt.title("confusion (row-normalised)"); plt.colorbar(); plt.show()

In [ ]:
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)

def train_stage_b(model, epochs):
    opt, sch = make_optim(model, epochs, len(dl_seq_tr)); best, hist = 1e9, []
    iso_iter = iter(dl_iso_tr)
    for ep in range(1, epochs+1):
        model.train(); tot = 0
        for x in dl_seq_tr:
            x = to_dev(x)
            try: xi = to_dev(next(iso_iter))
            except StopIteration: iso_iter = iter(dl_iso_tr); xi = to_dev(next(iso_iter))
            with torch.autocast(DEVICE, enabled=use_amp):
                o = model(x["hand"], x["body"], x["face"], x["mask"])
                lp = o["logits_ctc"].float().log_softmax(-1).transpose(0, 1)          # (T,B,V)
                l_ctc = ctc_loss(lp, x["targets"], x["lens"], x["target_lens"])
                oi = model(xi["hand"], xi["body"], xi["face"], xi["mask"]); l_ce = ce_loss(oi["logits_cls"], xi["targets"])
                loss = l_ctc + cfg.CE_WEIGHT_STAGE_B * l_ce
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += l_ctc.item()
        va = eval_sequence(model, dl_seq_va); hist.append((ep, tot/len(dl_seq_tr), va["wer"], va["cer"], va["sent_acc"]))
        if va["wer"] < best: best = va["wer"]; torch.save(model.state_dict(), cfg.OUT_DIR/"stageB_best.pt")
        if ep % 5 == 0 or ep == 1: print(f"[B] ep{ep:3d} ctc={tot/len(dl_seq_tr):.3f}  val WER={va['wer']:.3f}  CER={va['cer']:.3f}  sentAcc={va['sent_acc']:.3f}")
    model.load_state_dict(torch.load(cfg.OUT_DIR/"stageB_best.pt", map_location=DEVICE)); return pd.DataFrame(hist, columns=["ep","ctc","val_wer","val_cer","val_sent_acc"])

t0 = time.time(); histB = train_stage_b(model, cfg.EPOCHS_B); print(f"stage B done in {(time.time()-t0)/60:.1f} min")
histB.plot(x="ep", y=["val_wer","val_cer"], figsize=(8,3), title="Stage B"); plt.show()

In [ ]:
# ---- Stage B evaluation: WER / CER + Face ablation
resB      = eval_sequence(model, dl_seq_te, use_face=True)
resB_nof  = eval_sequence(model, dl_seq_te, use_face=False)
resA_post = eval_isolated(model, dl_iso_te)          # isolated หลัง stage B (ไม่ควรตกมาก)
print(f"Sign sequence TEST   WER={resB['wer']:.4f}  CER={resB['cer']:.4f}  sentence-acc={resB['sent_acc']:.3f}")
print(f"Face ablation (zero) WER={resB_nof['wer']:.4f}  CER={resB_nof['cer']:.4f}  → ΔWER={resB_nof['wer']-resB['wer']:+.4f}")
print(f"Isolated after B     acc={resA_post['acc']:.4f}  macro-F1={resA_post['macro_f1']:.4f}")
for r, h in list(zip(resB["refs"], resB["hyps"]))[:8]:
    print(("✓" if r == h else "✗"), "ref:", " ".join(r), " | hyp:", " ".join(h))
torch.save({"state_dict": model.state_dict(), "classes": SIGN_CLASSES, "cfg": {k: str(v) for k, v in asdict(cfg).items() if "KEY" not in k}},
           cfg.OUT_DIR / "sign_encoder_demo1.pt")

## 8. Video → Landmarks (MediaPipe Holistic) สำหรับวิดีโอใหม่

ต้องให้ output **schema เดียวกับ CSV ใน dataset** (คอลัมน์เดียวกัน, ~10 fps, จุดเดียวกัน: pose 11–16, face 105/70/300/334/61/291)  
รองรับทั้ง MediaPipe Tasks API (`HolisticLandmarker`, mediapipe ≥ 0.10.14) และ legacy `mp.solutions.holistic`

In [ ]:
import cv2, urllib.request
import mediapipe as mp

POSE_IDX = [11, 12, 13, 14, 15, 16]                       # l_sh, r_sh, l_el, r_el, l_wr, r_wr
FACE_IDX = [105, 70, 300, 334, 61, 291]                   # ตามลำดับคอลัมน์ lbrow_outer,lbrow_inner,rbrow_inner,rbrow_outer,mouth_right,mouth_left (ยืนยันด้วย sanity-check ด้านล่าง)
HOLISTIC_TASK_URL = "https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task"

class HolisticExtractor:
    "วิดีโอ → DataFrame คอลัมน์เดียวกับ dataset (frame, t_ms, lh_*, rh_*, pose+vis/pres, face)"
    def __init__(self, target_fps=cfg.TARGET_FPS):
        self.target_fps = target_fps
        self.legacy = hasattr(mp, "solutions") and hasattr(mp.solutions, "holistic")
        if self.legacy:
            self.h = mp.solutions.holistic.Holistic(model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)
        else:
            from mediapipe.tasks import python as mpp
            from mediapipe.tasks.python import vision
            task = cfg.OUT_DIR / "holistic_landmarker.task"
            if not task.exists(): urllib.request.urlretrieve(HOLISTIC_TASK_URL, task)
            self.vision = vision
            self.h = vision.HolisticLandmarker.create_from_options(vision.HolisticLandmarkerOptions(
                base_options=mpp.BaseOptions(model_asset_path=str(task)), running_mode=vision.RunningMode.VIDEO,
                min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5, min_hand_landmarks_confidence=0.5,
                min_face_detection_confidence=0.5, min_face_landmarks_confidence=0.5))
        print("MediaPipe backend:", "legacy solutions.holistic" if self.legacy else "Tasks HolisticLandmarker")

    def _detect(self, rgb, ts_ms):
        if self.legacy:
            r = self.h.process(rgb); g = lambda lm: (lm.landmark if lm else None)
            return g(r.left_hand_landmarks), g(r.right_hand_landmarks), g(r.pose_landmarks), g(r.face_landmarks)
        r = self.h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(ts_ms))
        g = lambda lm: (lm if lm else None)
        return g(r.left_hand_landmarks), g(r.right_hand_landmarks), g(r.pose_landmarks), g(r.face_landmarks)

    def extract(self, video_path, max_seconds=None):
        cap = cv2.VideoCapture(str(video_path)); src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step_ms = 1000.0 / self.target_fps; next_ms = 0.0                     # time-based sampling → ได้ target_fps เป๊ะไม่ว่ากล้อง 24/25/30/60 fps
        rows, fi, kept, t0 = [], 0, 0, time.time()
        while True:
            ok, frame = cap.read()
            if not ok: break
            ts_ms = 1000.0 * fi / src_fps
            if max_seconds and ts_ms > max_seconds*1000: break
            if ts_ms + 1e-6 >= next_ms:
                next_ms += step_ms
                lh, rh, pose, face = self._detect(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), ts_ms)
                row = {"frame": kept, "t_ms": int(ts_ms)}
                for name, hand in (("lh", lh), ("rh", rh)):
                    for i in range(21):
                        p = hand[i] if hand else None
                        row[f"{name}_x{i}"], row[f"{name}_y{i}"], row[f"{name}_z{i}"] = (p.x, p.y, p.z) if p else (np.nan,)*3
                for name, j in zip(POSE_PTS, POSE_IDX):
                    p = pose[j] if pose else None
                    row[f"{name}_x"], row[f"{name}_y"], row[f"{name}_z"] = (p.x, p.y, p.z) if p else (np.nan,)*3
                    row[f"{name}_vis"]  = (getattr(p, "visibility", np.nan) if p else np.nan)
                    row[f"{name}_pres"] = (getattr(p, "presence",   np.nan) if p else np.nan)
                for name, j in zip(FACE_PTS, FACE_IDX):
                    p = face[j] if face else None
                    row[f"{name}_x"], row[f"{name}_y"], row[f"{name}_z"] = (p.x, p.y, p.z) if p else (np.nan,)*3
                rows.append(row); kept += 1
            fi += 1
        cap.release(); dt = time.time() - t0
        df = pd.DataFrame(rows, columns=ALL_COLS)
        df.attrs.update(dict(src_fps=src_fps, n_src_frames=fi, n_kept=kept, extract_sec=dt, fps_processed=kept/max(dt,1e-6)))
        return df

extractor = HolisticExtractor()

def check_video_quality(df: pd.DataFrame, verbose=True) -> dict:
    "ตรวจคุณภาพ input ก่อน inference (กล้องจริงไม่เป๊ะ) → warnings ที่ actionable"
    det = lambda cols: float((~df[cols[0]].isna()).mean())
    sw = np.linalg.norm(df[["l_shoulder_x","l_shoulder_y"]].values - df[["r_shoulder_x","r_shoulder_y"]].values, axis=1)
    wrist_below = float(((df["l_wrist_y"] > 1.0) | (df["r_wrist_y"] > 1.0)).mean())
    q = dict(frames=len(df), pose=det(POSE_COLS), face=det(FACE_COLS), lh=det(LH_COLS), rh=det(RH_COLS),
             any_hand=float((~df["lh_x0"].isna() | ~df["rh_x0"].isna()).mean()),
             shoulder_width=float(np.nanmean(sw)), wrists_out_of_frame=wrist_below, warnings=[])
    if q["pose"] < 0.9: q["warnings"].append("pose detect < 90% — ให้เห็นไหล่ทั้งสองข้างชัด/แสงพอ")
    if q["face"] < 0.9: q["warnings"].append("face detect < 90% — หันหน้าตรง")
    if q["any_hand"] < 0.5: q["warnings"].append("เห็นมือ < 50% ของคลิป — กรอบภาพควรเห็นถึงเอว/มือตอนพัก")
    if q["wrists_out_of_frame"] > 0.3: q["warnings"].append("ข้อมือหลุดกรอบล่าง > 30% — ถอยกล้อง/เลื่อนกรอบลง")
    if not (0.12 < q["shoulder_width"] < 0.6): q["warnings"].append(f"ตัวเล็ก/ใหญ่ผิดปกติ (shoulder width={q['shoulder_width']:.2f}) — ระยะกล้องควรให้ไหล่กว้าง ~15-50% ของเฟรม")
    if len(df) < 15: q["warnings"].append("คลิปสั้นเกิน (<1.5 s)")
    if verbose:
        print(f"quality: frames={q['frames']} pose={q['pose']:.2f} face={q['face']:.2f} LH={q['lh']:.2f} RH={q['rh']:.2f} shoulderW={q['shoulder_width']:.2f}")
        for w in q["warnings"]: print("  ⚠", w)
        if not q["warnings"]: print("  ✓ input OK")
    return q

In [ ]:
# ---- Sanity check: extractor ของเรา vs landmark CSV ใน dataset (คลิปเดียวกัน)
def _corr(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return np.corrcoef(a[m], b[m])[0, 1] if m.sum() > 10 else np.nan

def compare_with_dataset(video_path):
    ours = extractor.extract(video_path); ref_p = resolve_landmark(meta_sent.loc[meta_sent[vcol].map(lambda s: Path(s).name) == Path(video_path).name, lp_col].iloc[0])
    ref = pd.read_csv(ref_p); n = min(len(ours), len(ref))
    print(f"{Path(video_path).name}: ours {len(ours)} frames vs dataset {len(ref)} frames | extract {ours.attrs['fps_processed']:.1f} fps")
    for grp, cols in [("LH", LH_COLS[:6]), ("RH", RH_COLS[:6]), ("POSE", POSE_COLS), ("FACE", FACE_COLS)]:
        cs = [_corr(ours[c].values[:n], ref[c].values[:n]) for c in cols]
        print(f"   {grp:5s} mean corr = {np.nanmean(cs):.3f}")
    return ours, ref

if sample_videos:
    ours, ref = compare_with_dataset(sample_videos[0])
    # ถ้า FACE corr ต่ำ (<0.5) แต่ POSE สูง → ลองสลับ outer/inner: FACE_IDX = [70,105,334,300,61,291] แล้วรัน cell นี้ใหม่

## 9. LLM — Sign tokens + Face cue → Natural Thai

- Vision model ให้ **gloss sequence** (ลำดับแบบภาษามือ: กรรม-ประธาน-กริยา) + **face cue** (จาก geometry ของคิ้ว/ปาก เทียบกับ baseline ของ dataset)
- LLM มีหน้าที่แค่เรียบเรียงเป็นภาษาไทยธรรมชาติ ห้ามเพิ่มความหมายใหม่
- ไม่มี API key → ใช้ `rule_gloss_to_thai` (ใช้เป็น reference สำหรับ BLEU/chrF ด้วย — เป็น proxy, ผลจริงต้องดู human eval)

In [ ]:
# ---- face cue (heuristic จาก face features เทียบ baseline isolated-train)
_geo = np.concatenate([it["feat"]["face"][:, -3:] for it in iso_train], 0)
FACE_BASE_MEAN, FACE_BASE_STD = _geo.mean(0), _geo.std(0) + 1e-6

def face_cue(face_feat: np.ndarray) -> dict:
    z = (face_feat[:, -3:] - FACE_BASE_MEAN) / FACE_BASE_STD           # (T,3): brow_raise, mouth_width, brow_gap
    peak = np.percentile(z, 80, axis=0); cue = []
    if peak[0] > 1.0: cue.append("eyebrows raised (question-like / emphasis)")
    if peak[0] < -1.0: cue.append("eyebrows lowered / furrowed (negative or wh-question)")
    if peak[1] > 1.0: cue.append("mouth widened (smile / positive)")
    if np.abs(z).max() < 0.7: cue.append("neutral")
    return dict(text=", ".join(cue) or "neutral", z_peak=peak.round(2).tolist())

# ---- rule-based gloss → Thai (fallback + proxy reference)
SUBJ  = {"แม่","คุณ","แฟน","พี่","น้อง","ฉัน","ยาย","แมว"}
OBJ   = {"ภาษามือ","ขนมปัง","ข้าว","ไข่"}
PLACE = {"กรุงเทพ","โรงเรียน","บ้าน","ตลาด"}
GREET = {"สวัสดี","ขอโทษ","ขอบคุณ"}
TIME  = {"วันนี้","เช้า","วันหยุด","พรุ่งนี้"}
QWORD = {"ทำไม","อะไร","ที่ไหน"}
STATE = {"เหงา","โสด","โกรธ","กลัว","เหนื่อย","สบายดี","หูหนวก","ง่วง","ร้อน","ดี","อยู่บ้าน","ทำงาน"}

def rule_gloss_to_thai(words):
    ws = list(dict.fromkeys(w for w in words if w != "<null>"))   # dedupe (กัน CTC ทำนายซ้ำ)
    greet=[w for w in ws if w in GREET]; tm=[w for w in ws if w in TIME]; subj=[w for w in ws if w in SUBJ]
    obj=[w for w in ws if w in OBJ]; place=[w for w in ws if w in PLACE]; q=[w for w in ws if w in QWORD]
    state=[w for w in ws if w in STATE]
    verbs=[w for w in ws if w not in GREET|TIME|SUBJ|OBJ|PLACE|QWORD|STATE]
    S = subj[0] if subj else ""; O2 = subj[1] if len(subj) > 1 else ""
    out = "".join(greet) + ("  " if greet else "") + "".join(tm) + S
    if O2 and "แต่งงาน" in verbs:  return out + "แต่งงานกับ" + O2
    if O2 and not verbs and not state: return out + "เป็นแฟนกับ" + O2 if "แฟน" in (S, O2) else out + O2
    if "เกิด" in verbs and "หูหนวก" in state: return out + "เกิดมาหูหนวก"
    core = ""
    if "ชอบ" in verbs: core += "ชอบ"
    if q and "ที่ไหน" in q and tm: core += "จะ"
    if "ไป" in verbs: core += "ไป"
    if "เที่ยว" in verbs: core += "เที่ยว"
    core += "".join(v for v in verbs if v not in ("ชอบ","ไป","เที่ยว"))
    core += "".join(obj) + "".join(place) + "".join(state)
    if q: core += "".join(q) + "?"
    return out + core

for g in [["ขนมปัง","ฉัน","กิน"], ["กรุงเทพ","ยาย","เที่ยว","ชอบ"], ["พรุ่งนี้","คุณ","ข้าว","กิน","ไป","ที่ไหน"], ["คุณ","ฉัน","แต่งงาน"], ["ฉัน","เกิด","หูหนวก"]]:
    print(g, "→", rule_gloss_to_thai(g))

In [ ]:
LLM_SYSTEM = (
 "You are a Thai Sign Language (TSL) to Thai interpreter. Input is a gloss sequence in TSL word order "
 "(typically topic/object first, verb last, no function words) plus a facial-expression cue. "
 "Rewrite as ONE natural, grammatical Thai sentence with the same meaning. Rules: do NOT add facts not in the glosses; "
 "you MAY add function words (กับ, ที่, จะ, ไป, มา, เป็น, ไหม) and reorder; if the face cue is question-like or a question word exists, end with '?'. "
 "Output only the Thai sentence."
)

def llm_translate(glosses_words, cue_text="neutral"):
    "→ (thai_sentence, provider_used)"
    words = [w for w in glosses_words if w != "<null>"]
    if not words: return "", "none"
    if cfg.LLM_PROVIDER == "openai" and os.environ.get("OPENAI_API_KEY"):
        try:
            from openai import OpenAI
            client = OpenAI()
            r = client.responses.create(model=cfg.LLM_MODEL, instructions=LLM_SYSTEM,
                                        input=f"GLOSSES: {' | '.join(words)}\nFACE CUE: {cue_text}")
            return r.output_text.strip(), cfg.LLM_MODEL
        except Exception as e:
            print("LLM API error → rule fallback:", type(e).__name__, str(e)[:120])
    return rule_gloss_to_thai(words), "rule"

print(llm_translate(["คุณ","ชื่อ","อะไร"], "eyebrows raised (question-like)"))

In [ ]:
# ---- Translation eval (BLEU / chrF) บน test sentences ของ Stage B
# reference = rule-based ของ gloss จริง (แก้/เพิ่ม human reference ได้ใน REFERENCE_OVERRIDES)
REFERENCE_OVERRIDES = {  # "pattern (space-joined words)": "natural Thai reference"
    # "ขนมปัง ฉัน กิน": "ฉันกินขนมปัง",
}
def reference_for(ref_words):
    return REFERENCE_OVERRIDES.get(" ".join(ref_words), rule_gloss_to_thai(ref_words))

refs, hyps_pred, hyps_oracle = [], [], []
for it, ref_w, hyp_w in zip(sent_test, resB["refs"], resB["hyps"]):
    cue = face_cue(it["feat"]["face"])["text"]
    refs.append(reference_for(ref_w))
    hyps_pred.append(llm_translate(hyp_w, cue)[0])       # end-to-end (จาก vision prediction)
    hyps_oracle.append(llm_translate(ref_w, cue)[0])     # oracle glosses (วัด LLM อย่างเดียว)
bleu_e2e, chrf_e2e = bleu_chrf(refs, hyps_pred); bleu_or, chrf_or = bleu_chrf(refs, hyps_oracle)
print(f"Translation (oracle glosses)  BLEU={bleu_or:.2f}  chrF={chrf_or:.2f}")
print(f"Translation (end-to-end)      BLEU={bleu_e2e:.2f} chrF={chrf_e2e:.2f}")
pd.DataFrame({"ref": refs, "e2e": hyps_pred, "oracle": hyps_oracle}).head(10)

## 10. TTS — Natural Thai → .wav

In [ ]:
from scipy.io import wavfile
from IPython.display import Audio, display

_mms = {}
def tts_mms(text, out_path):
    "open-source: facebook/mms-tts-tha (VITS) — ถ้า tokenizer ต้องการ uroman จะ romanize อัตโนมัติ (pip install uroman)"
    if not _mms:
        from transformers import VitsModel, AutoTokenizer
        _mms["tok"] = AutoTokenizer.from_pretrained("facebook/mms-tts-tha"); _mms["m"] = VitsModel.from_pretrained("facebook/mms-tts-tha").eval()
    tok, m = _mms["tok"], _mms["m"]
    if getattr(tok, "is_uroman", False):
        import uroman as ur; text = ur.Uroman().romanize_string(text)
    with torch.no_grad(): wav = m(**tok(text, return_tensors="pt")).waveform[0].numpy()
    wavfile.write(out_path, m.config.sampling_rate, (wav * 32767).astype(np.int16)); return out_path

def tts_openai(text, out_path):
    from openai import OpenAI
    r = OpenAI().audio.speech.create(model=cfg.OPENAI_TTS_MODEL, voice=cfg.OPENAI_TTS_VOICE, input=text, response_format="wav")
    Path(out_path).write_bytes(r.content); return out_path

def tts(text, out_path):
    if cfg.TTS_BACKEND == "openai" and os.environ.get("OPENAI_API_KEY"): return tts_openai(text, out_path)
    return tts_mms(text, out_path)

wav_path = tts("ฉันกินข้าว", cfg.OUT_DIR / "tts_test.wav"); display(Audio(str(wav_path)))

## 11. End-to-End inference ด้วยวิดีโอ .mp4 ของคุณ

ใส่ path วิดีโอใน `MY_VIDEO` (ถ่ายหน้าตรง เห็นไหล่ทั้งสองข้าง แสงพอ) และถ้ารู้ประโยคที่ทำ ให้ใส่ `EXPECTED_WORDS` เพื่อคำนวณ WER ทันที

**ความยืดหยุ่นกับกล้องจริง (สิ่งที่ pipeline นี้จัดการให้แล้ว)**
| ปัญหาจริง | วิธีรับมือใน notebook |
|---|---|
| fps กล้องไม่ตรง (24/25/30/60) | `HolisticExtractor` sample ตามเวลา → 10 fps เท่า dataset เสมอ |
| ความละเอียด / ระยะกล้อง / ตำแหน่งในเฟรม | MediaPipe ให้พิกัด normalized + เรา normalize ด้วยกึ่งกลาง+ความกว้างไหล่ |
| กล้องเอียง / มุมต่างเล็กน้อย / คนถนัดซ้าย | `augment()`: rotation ±12°, scale, shift, **mirror+swap L/R** |
| MediaPipe จับมือไม่ได้บางเฟรม (แสง, มือบัง, มือหลุดเฟรม) | presence flag + `hand_drop` augmentation |
| ท่าที่ไม่ใช่ 51 คำ / ท่าพัก | class `null_act` + `MIN_TOKEN_CONF` ทิ้ง token ที่ CTC ไม่มั่นใจ (ไม่ force เป็นคำใกล้เคียง) |
| input แย่จนไม่ควรเชื่อผล | `check_video_quality()` เตือนก่อนแปล (มือหลุดเฟรม, ตัวเล็กเกิน, หน้าไม่ตรง) |

> ข้อจำกัดที่ต้องรู้: ระบบ **รู้จักแค่ 51 คำ / 76 pattern** — ประโยคอื่นจะได้แค่คำที่ใกล้ที่สุดหรือถูกทิ้ง; ความแม่นยำกับผู้ใช้ใหม่จะขึ้นกับว่าท่าตรงกับ signer ใน dataset แค่ไหน (ดูวิดีโออ้างอิงใน `videos/user_sign/` ก่อนถ่าย)

In [ ]:
from IPython.display import Video

MIN_TOKEN_CONF = 0.5     # token ที่ CTC มั่นใจน้อยกว่านี้จะถูกทิ้ง (ป้องกันท่าที่ไม่ใช่ 51 คำ ถูก force เป็นคำใกล้เคียง) — ปรับตาม val set

@torch.no_grad()
def ctc_decode_with_confidence(logits_ctc, lens, drop_null=True):
    "greedy CTC + confidence ต่อ token = max prob เฉลี่ยของเฟรมที่ emit token นั้น"
    probs = logits_ctc.softmax(-1).cpu().numpy(); ids_all, conf_all = [], []
    for p, L in zip(probs, lens):
        am = p[:L].argmax(-1); seq, cf, prev, cur = [], [], BLANK, []
        for t, tok in enumerate(am):
            if tok != BLANK and tok != prev: 
                if cur: cf.append(float(np.mean(cur))); cur = []
                seq.append(int(tok) - 1)
            if tok != BLANK: cur.append(p[t, tok])
            prev = tok
        if cur: cf.append(float(np.mean(cur)))
        pairs = [(i, c) for i, c in zip(seq, cf) if not (drop_null and SIGN_CLASSES[i] == NULL_CLASS)]
        ids_all.append([i for i, _ in pairs]); conf_all.append([c for _, c in pairs])
    return ids_all, conf_all

@torch.no_grad()
def run_pipeline(video_path, expected_words=None, show=True):
    model.eval(); lat = {}
    t = time.time(); df = extractor.extract(video_path); lat["1_mediapipe_s"] = time.time() - t
    t = time.time(); feat = make_features(load_landmarks(df))
    if len(feat["hand"]) > cfg.MAX_FRAMES_SEQ: feat = temporal_resample(feat, cfg.MAX_FRAMES_SEQ)
    x = collate([(feat, [0])]); x = to_dev(x)
    o = model(x["hand"], x["body"], x["face"], x["mask"])
    ids, confs = ctc_decode_with_confidence(o["logits_ctc"], x["lens"].tolist())
    ids, confs = ids[0], confs[0]
    keep = [(i, c) for i, c in zip(ids, confs) if c >= MIN_TOKEN_CONF]           # ต่ำกว่า threshold → ทิ้ง (ถือว่านอก 51 คำ / ไม่มั่นใจ)
    rejected = [(SIGN_CLASSES[i], round(c, 2)) for i, c in zip(ids, confs) if c < MIN_TOKEN_CONF]
    ids = [i for i, _ in keep]
    glosses = [SIGN_CLASSES[i] for i in ids]; words = ids_to_words(ids); lat["2_sign_encoder_s"] = time.time() - t
    quality = check_video_quality(df, verbose=show)
    cue = face_cue(feat["face"])
    t = time.time(); thai, prov = llm_translate(words, cue["text"]); lat["3_llm_s"] = time.time() - t
    t = time.time(); wav = tts(thai, cfg.OUT_DIR / (Path(video_path).stem + "_tts.wav")) if thai else None; lat["4_tts_s"] = time.time() - t
    lat["total_s"] = sum(lat.values()); lat["mediapipe_fps"] = df.attrs["fps_processed"]

    res = dict(video=str(video_path), n_frames=len(df), glosses=glosses, words=words, token_conf=[round(c, 2) for _, c in keep],
               rejected_low_conf=rejected, face_cue=cue["text"], thai=thai, llm=prov, wav=str(wav) if wav else None,
               quality=quality, latency=lat)
    if expected_words:
        res["wer"], res["cer"] = wer_cer([expected_words], [words])
    if show:
        print(json.dumps({k: v for k, v in res.items() if k not in ("latency", "quality")}, ensure_ascii=False, indent=1))
        print("latency:", {k: round(v, 2) for k, v in lat.items()})
        print("\n  SIGNS :", words, "\n  FACE  :", cue["text"], "\n  THAI  :", thai)
        if wav: display(Audio(str(wav)))
    return res

MY_VIDEO = "data_test/ไปทานด้วยกันมั้ย.mp4"            # <-- ใส่ path วิดีโอของคุณ
EXPECTED_WORDS = None                      # เช่น ["ฉัน","ข้าว","กิน"] (ลำดับ gloss แบบภาษามือ) หรือ None

if Path(MY_VIDEO).exists():
    display(Video(MY_VIDEO, width=360)); result = run_pipeline(MY_VIDEO, EXPECTED_WORDS)
elif sample_videos:
    print("ไม่พบ MY_VIDEO → demo ด้วยวิดีโอตัวอย่างจาก dataset แทน")
    exp = parse_sentence_glosses(sample_videos[0].stem); result = run_pipeline(sample_videos[0], [CLASS2WORD[g] for g in exp])

In [ ]:
# รันหลายวิดีโอของคุณเองทีเดียว (โฟลเดอร์) → ตาราง + WER รวม
MY_VIDEO_DIR = Path("./my_videos")          # วางไฟล์ .mp4 ; ถ้ารู้ประโยค ให้ตั้งชื่อไฟล์เป็น gloss คั่นด้วย __ เช่น ฉัน__ข้าว__กิน_1.mp4
rows = []
for v in sorted(MY_VIDEO_DIR.glob("*.mp4")) if MY_VIDEO_DIR.exists() else []:
    exp = [CLASS2WORD.get(g, class_to_word(g)) for g in parse_sentence_glosses(v.stem)] if "__" in v.stem else None
    r = run_pipeline(v, exp, show=False)
    rows.append(dict(video=v.name, expected=" ".join(exp) if exp else "", predicted=" ".join(r["words"]), thai=r["thai"],
                     face=r["face_cue"], wer=r.get("wer"), latency_s=round(r["latency"]["total_s"], 2)))
if rows:
    df_my = pd.DataFrame(rows); display(df_my)
    if df_my["wer"].notna().any(): print("mean WER on my videos:", round(df_my["wer"].mean(), 3))
    df_my.to_csv(cfg.OUT_DIR / "my_videos_results.csv", index=False)

## 12. Human evaluation templates (TTS MOS / E2E semantic accuracy) + Metrics summary

MOS และ semantic accuracy ต้องมีคนให้คะแนน → สร้าง CSV ให้กรอก แล้วรัน cell สรุปอีกครั้ง

In [ ]:
# template สำหรับให้คน rate
tts_rows = []
for i, (w, th) in enumerate(zip(resB["hyps"][:10], hyps_pred[:10])):
    if not th: continue
    p = tts(th, cfg.OUT_DIR / f"eval_tts_{i}.wav")
    tts_rows.append(dict(id=i, wav=str(p), thai=th, mos_naturalness_1to5="", mos_intelligibility_1to5=""))
pd.DataFrame(tts_rows).to_csv(cfg.OUT_DIR / "human_eval_tts_MOS.csv", index=False)

e2e_rows = [dict(id=i, ref_glosses=" ".join(r), pred_glosses=" ".join(h), ref_thai=rf, pred_thai=ph,
                 semantic_correct_0or1="", fluency_1to5="", notes="")
            for i, (r, h, rf, ph) in enumerate(zip(resB["refs"], resB["hyps"], refs, hyps_pred))]
pd.DataFrame(e2e_rows).to_csv(cfg.OUT_DIR / "human_eval_e2e.csv", index=False)
print("saved:", cfg.OUT_DIR / "human_eval_tts_MOS.csv", "|", cfg.OUT_DIR / "human_eval_e2e.csv")

def load_human(name, col):
    p = cfg.OUT_DIR / name
    if p.exists():
        s = pd.to_numeric(pd.read_csv(p)[col], errors="coerce").dropna()
        if len(s): return round(s.mean(), 3)
    return "pending (fill CSV)"

In [ ]:
lat = result["latency"] if "result" in globals() else {}
summary = pd.DataFrame([
    ["Sign Recognition", "Accuracy",        f"{resA_post['acc']:.4f}"],
    ["Sign Recognition", "Macro-F1",        f"{resA_post['macro_f1']:.4f}"],
    ["Sign Sequence",    "WER",             f"{resB['wer']:.4f}"],
    ["Sign Sequence",    "CER",             f"{resB['cer']:.4f}"],
    ["Translation",      "BLEU (char) e2e / oracle", f"{bleu_e2e:.2f} / {bleu_or:.2f}"],
    ["Translation",      "chrF e2e / oracle",        f"{chrf_e2e:.2f} / {chrf_or:.2f}"],
    ["Face contribution","ΔWER (no-face − face)",    f"{resB_nof['wer']-resB['wer']:+.4f}"],
    ["TTS",              "MOS naturalness",          load_human("human_eval_tts_MOS.csv", "mos_naturalness_1to5")],
    ["End-to-End",       "Semantic accuracy",        load_human("human_eval_e2e.csv", "semantic_correct_0or1")],
    ["Runtime",          "MediaPipe FPS",            f"{lat.get('mediapipe_fps', float('nan')):.1f}"],
    ["Runtime",          "Total latency (s)",        f"{lat.get('total_s', float('nan')):.2f}"],
], columns=["Stage", "Metric", "Value"])
display(summary); summary.to_csv(cfg.OUT_DIR / "metrics_summary.csv", index=False)
print("artifacts in", cfg.OUT_DIR.resolve())

### Next steps (หลัง Demo 1)
- เปิด `USE_EXPERT_AUGMENTED=True` เมื่อมี GPU/RAM พอ (+32k คลิป) — ช่วย isolated accuracy กับ signer ใหม่
- Beam-search CTC + gloss language model (n-gram บน 76 patterns) ลด WER
- Train emotion / non-manual classifier แทน heuristic `face_cue`
- ใส่ human reference ใน `REFERENCE_OVERRIDES` ให้ครบ 76 ประโยค แล้ว BLEU/chrF จะมีความหมายจริง
- Real-time: sliding window บน webcam (feature pipeline เดิม, ตัด null_act ระหว่าง sign)